In [1]:
import os
import requests
from bs4 import BeautifulSoup
import jieba
from collections import Counter
import numpy as np
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from PIL import Image
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}
url = 'http://www.hprc.org.cn/wxzl/wxysl/lczf/shisijbg/202504/t20250408_5867286.html'
response = requests.get(url, headers=headers)
response.encoding = 'utf-8' 
soup = BeautifulSoup(response.text, 'html.parser')
title_element = soup.find('h1')
if title_element:
    title = title_element.text.strip() 
    print(f"标题: {title}")
else:
    print("未找到标题元素")
    title = "未知标题"
content_element = soup.find('div', class_='TRS_Editor')
if content_element:
    content = content_element.text.strip() 
    print("正文内容已提取")
else:
    print("未找到正文内容元素")
    content = "未知内容"
current_directory = os.getcwd()
print(f"当前工作目录: {current_directory}")
file_path = os.path.join(current_directory, '2025年政府工作报告.txt')
with open(file_path, 'w', encoding='utf-8') as f:
    f.write(title + '\n\n' + content)
print(f'爬取完成，内容已写入 {file_path} 文件')
with open(file_path, 'r', encoding='utf-8') as f:
    text = f.read()
words = jieba.lcut(text)
word_counts = Counter(words)
print("词频统计结果：")
for word, count in word_counts.most_common(20):
    print(f"{word}: {count}")

Building prefix dict from the default dictionary ...


Loading model from cache C:\Users\32517\AppData\Local\Temp\jieba.cache


未找到标题元素
正文内容已提取
当前工作目录: C:\Users\32517\Desktop\4
爬取完成，内容已写入 C:\Users\32517\Desktop\4\2025年政府工作报告.txt 文件


Loading model cost 0.543 seconds.


Prefix dict has been built successfully.


词频统计结果：
，: 716
。: 492
、: 341
和: 214
 : 162
的: 145
发展: 134

: 88
推进: 82
建设: 71
等: 60
加强: 56
推动: 55
“: 54
”: 54
加快: 53
经济: 52
完善: 52
新: 50
实施: 43


In [2]:
import jieba
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from collections import Counter
import imageio.v2 as imageio
import matplotlib
import pygame
import warnings
import os
import sys
warnings.filterwarnings('ignore')
matplotlib.use('TkAgg')
plt.rcParams['font.family'] = 'SimHei'


def process_text_and_generate_wordcloud(txt_file_path, mask_image_path, output_image_path='china_wordcloud.png'):
    # 检查文件是否存在
    if not os.path.exists(txt_file_path):
        print(f"错误: 文件 {txt_file_path} 不存在")
        return None
    if not os.path.exists(mask_image_path):
        print(f"错误: 图片文件 {mask_image_path} 不存在")
        return None
    
    with open(txt_file_path, 'r', encoding='utf-8') as file:
        text = file.read()

    words = jieba.lcut(text)

    word_counts = Counter(words)

    stop_words = {'的', '了', '在', '是', '我', '有', '和', '就', '不', '人', '都', '一', '一个', '上', '也', '很', '到', '说', '要', '去',
                  '你', '会', '着', '没有', '看', '好', '自己', '这', '那'}

    filtered_words = {}
    for word, count in word_counts.items():
        if len(word) > 1 and word not in stop_words and not word.isspace():
            filtered_words[word] = count

    sorted_words = sorted(filtered_words.items(), key=lambda x: x[1], reverse=True)

    print("前20个高频词:")
    for i, (word, count) in enumerate(sorted_words[:20], 1):
        print(f"{i:2d}. {word}: {count}")

    china_mask = imageio.imread(mask_image_path)

    # 尝试找到合适的字体文件
    font_path = None
    
    # 首先检查当前目录
    possible_fonts = ['simhei.ttf', 'SimHei.ttf', 'simhei.TTF']
    for font in possible_fonts:
        if os.path.exists(font):
            font_path = font
            break
    
    # 如果当前目录没有，检查系统字体目录
    if font_path is None:
        if sys.platform == 'win32':
            # Windows 系统字体目录
            system_fonts = [
                'C:\\Windows\\Fonts\\simhei.ttf',
                'C:\\Windows\\Fonts\\SimHei.ttf',
                'C:\\Windows\\Fonts\\msyh.ttc',  # 微软雅黑
                'C:\\Windows\\Fonts\\msyh.ttf',
            ]
            for font in system_fonts:
                if os.path.exists(font):
                    font_path = font
                    break
        elif sys.platform == 'darwin':
            # macOS 系统字体目录
            system_fonts = [
                '/Library/Fonts/SimHei.ttf',
                '/System/Library/Fonts/PingFang.ttc',
            ]
            for font in system_fonts:
                if os.path.exists(font):
                    font_path = font
                    break
        else:
            # Linux 系统字体目录
            system_fonts = [
                '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc',
                '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
            ]
            for font in system_fonts:
                if os.path.exists(font):
                    font_path = font
                    break
    
    wordcloud = WordCloud(
        font_path=font_path,
        width=800,
        height=600,
        background_color='white',
        mask=china_mask,
        max_words=200,
        max_font_size=100,
        min_font_size=10,
        random_state=42,
        collocations=False
    )

    word_freq_dict = dict(sorted_words[:100])
    wordcloud.generate_from_frequencies(word_freq_dict)

    plt.figure(figsize=(12, 8))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title('中国地图形状词云图', fontsize=16, pad=20)

    plt.savefig(output_image_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"词云图已保存至: {output_image_path}")

    plt.show()

    return sorted_words


if __name__ == "__main__":
    txt_file = "2025年政府工作报告.txt"
    china_mask = "china.jpg"

    result = process_text_and_generate_wordcloud(txt_file, china_mask)


pygame 2.6.1 (SDL 2.28.4, Python 3.12.7)
Hello from the pygame community. https://www.pygame.org/contribute.html
前20个高频词:
 1. 发展: 134
 2. 推进: 82
 3. 建设: 71
 4. 加强: 56
 5. 推动: 55
 6. 加快: 53
 7. 经济: 52
 8. 完善: 52
 9. 实施: 43
10. 支持: 43
11. 政策: 41
12. 改革: 36
13. 政府: 35
14. 创新: 35
15. 服务: 35
16. 提升: 33
17. 扩大: 32
18. 促进: 32
19. 机制: 32
20. 制度: 31


词云图已保存至: china_wordcloud.png
